# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ramithnayak8/ML_pipeline/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

My question is "which page first?" -- a ranking problem built on a yes/no observed label
(`is_declining_label`). Per the toolkit: start with **Logistic Regression** (readable), then
**Random Forest** (stronger, if it earns its keep), evaluated by precision@K rather than plain
accuracy, since a ranking is only ever judged by its top slice. I am training both rather than
jumping straight to the more complex model -- simplicity is the default, and the comparison
table below is what decides whether the added complexity was worth it.

In [1]:
import numpy as np
import pandas as pd

raw = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")
df = raw[(raw["impressions_90d"] > 0) & (raw["content_age_days"] >= 90)].copy()
df = df.drop_duplicates(subset=["content_id"]).reset_index(drop=True)

# same feature build as w03_feature_leakage_check.ipynb
df["has_keyword_data"] = df["search_volume"].notna().astype(int)
df["has_word_count"] = df["word_count"].notna().astype(int)

numeric_fill_zero = ["search_volume", "competition", "cpc", "word_count", "char_count",
                     "impressions_90d", "clicks_90d", "sessions_90d", "ai_sessions_90d",
                     "days_with_impressions", "days_with_sessions", "content_age_days",
                     "days_since_last_update", "ctr", "avg_position", "engagement_rate",
                     "scroll_rate", "ai_traffic_pct"]
for c in numeric_fill_zero:
    df[c] = df[c].replace([np.inf, -np.inf], np.nan).fillna(0)

categorical_cols = ["competition_level", "content_type", "main_intent", "age_tier",
                    "freshness_tier", "word_count_tier", "impression_tier", "position_tier"]
for c in categorical_cols:
    df[c] = df[c].fillna("unknown").astype(str)

df["log_impressions_90d"] = np.log1p(df["impressions_90d"])
df["log_clicks_90d"] = np.log1p(df["clicks_90d"])
df["log_sessions_90d"] = np.log1p(df["sessions_90d"])
df["log_ai_sessions_90d"] = np.log1p(df["ai_sessions_90d"])

df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

feature_cols = (["search_volume", "competition", "cpc", "word_count", "char_count",
                 "log_impressions_90d", "log_clicks_90d", "log_sessions_90d", "log_ai_sessions_90d",
                 "days_with_impressions", "days_with_sessions", "content_age_days",
                 "days_since_last_update", "ctr", "avg_position", "engagement_rate",
                 "scroll_rate", "ai_traffic_pct", "has_keyword_data", "has_word_count"]
                + categorical_cols)

# same baseline rule as w04_baseline_score.ipynb, so the comparison is apples-to-apples
stale_visible = (df["days_since_last_update"] >= 180) & (df["impressions_90d"] >= 500)
low_ctr_visible = ((df["impressions_90d"] >= 500) & (df["avg_position"] > 0)
                   & (df["avg_position"] <= 20) & (df["ctr"] < 0.5))
thin_visible = (df["word_count"] > 0) & (df["word_count"] < 1200) & (df["impressions_90d"] >= 250)
reason_count = stale_visible.astype(int) + low_ctr_visible.astype(int) + thin_visible.astype(int)
df["rule_score"] = reason_count + 0.99 * df["impressions_90d"].rank(pct=True)

print(f"{len(df):,} rows, {len(feature_cols)} raw feature columns")


30,000 rows, 28 raw feature columns


## 2. Split design

**Client-grouped, 80/20.** Pages from the same client share hidden character (the same site,
the same SEO practices, the same content team) -- a random row-level split would let a model
partly memorize a client instead of learning a generalizable pattern. `GroupShuffleSplit` on
`client_id` guarantees zero client overlap between train and test, verified below. With only 32
clients total, a single split is a fairly coarse instrument (this is exactly the kind of thing
`w06_validation_audit.ipynb` digs into further) -- but it is the honest choice for this
question, and it is the same split I evaluate my Week-4 baseline rule on too.

In [2]:
from sklearn.model_selection import GroupShuffleSplit

X = pd.get_dummies(df[feature_cols], drop_first=True)
y = df["is_declining_label"]

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=df["client_id"]))

train_clients = set(df.loc[train_idx, "client_id"])
test_clients = set(df.loc[test_idx, "client_id"])

print(f"train: {len(train_idx):,} rows, {len(train_clients)} clients")
print(f"test:  {len(test_idx):,} rows, {len(test_clients)} clients")
print("client overlap between train and test:", len(train_clients & test_clients), "(0 = honest split)")


train: 23,837 rows, 25 clients
test:  6,163 rows, 7 clients
client overlap between train and test: 0 (0 = honest split)


## 3. Train + compare vs my baseline

Same test rows, same `precision_at_k` function, same metrics as `w04_baseline_score.ipynb` --
the baseline's `rule_score` gets evaluated on this exact held-out slice, not the full dataset,
so the comparison is fair. Logistic regression's inputs are standardized (trees do not need
that); both models use a fixed `random_state` for reproducibility.

In [3]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

logit = LogisticRegression(max_iter=2000, random_state=42).fit(X_train_scaled, y_train)
rf = RandomForestClassifier(n_estimators=300, max_depth=8, random_state=42,
                             class_weight="balanced_subsample").fit(X_train, y_train)

rule_test = df.loc[test_idx, "rule_score"].values
proba_logit = logit.predict_proba(X_test_scaled)[:, 1]
proba_rf = rf.predict_proba(X_test)[:, 1]

print(f"test base rate: {y_test.mean():.1%}\n")
rows = [
    ("baseline rule (w04)", rule_test),
    ("logistic_regression", proba_logit),
    ("random_forest", proba_rf),
]
for name, scores in rows:
    print(f"{name:>20}: P@20={precision_at_k(scores, y_test, 20):.3f}  "
          f"P@50={precision_at_k(scores, y_test, 50):.3f}  "
          f"ROC-AUC={roc_auc_score(y_test, scores):.3f}")


test base rate: 51.1%

 baseline rule (w04): P@20=0.400  P@50=0.420  ROC-AUC=0.532
 logistic_regression: P@20=0.700  P@50=0.720  ROC-AUC=0.615
       random_forest: P@20=0.550  P@50=0.640  ROC-AUC=0.605


## 4. Errors and interpretation

Both models clear the baseline by a wide margin on this split, but the finding I did not expect:
**logistic regression edges out random forest here**, on both P@20 and P@50 -- the opposite of
the reference pipeline's own result. With only 32 clients total (25 in training), a single
tree-ensemble draw has less to learn a stable pattern from than a linear model does, and which 7
clients land in the test fold can matter more than the choice of algorithm. That is a real,
observed result on this split and this seed -- not a general claim that "random forest is worse,"
which is exactly what `w06_validation_audit.ipynb` needs to stress-test with more than one split.

Random forest's top features are `days_with_impressions`, `log_impressions_90d`, and
`avg_position` -- all plausible (persistent, visible, well-positioned pages are the ones with
enough of a track record for the model to read a trend from), and none of them are suspiciously
perfect, which is itself a leakage sanity check. Reading the model's actual mistakes: several
of its wrong top-50 picks share a pattern -- decent visibility, weak position (20-40), and a flat
zero CTR, but the page is actually `stable` or `up`, not `down`. That suggests the model reads
"weak position and weak CTR" as risk in general, but that combination does not reliably predict
which DIRECTION a page is currently moving -- a real limit of these features for this label.

In [4]:
importances = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)
print("Top 5 random forest features:")
print(importances.head(5).round(4))

order = np.argsort(-proba_rf)
top50_idx = test_idx[order[:50]]
wrong_mask = (y.loc[top50_idx].values == 0)
print(f"\n{wrong_mask.sum()} of the random forest's top 50 test picks are NOT actually declining")

wrong_ids = df.loc[top50_idx][wrong_mask]
print(wrong_ids[["content_id", "trend_direction", "impressions_90d", "avg_position", "ctr",
                  "word_count"]].head(3).to_string(index=False))


Top 5 random forest features:
days_with_impressions    0.1545
log_impressions_90d      0.1243
avg_position             0.1083
content_age_days         0.0921
word_count               0.0534
dtype: float64

18 of the random forest's top 50 test picks are NOT actually declining
          content_id trend_direction  impressions_90d  avg_position  ctr  word_count
content_c148e44db30d              up              335          31.3  0.0      1622.0
content_0b47dae0c7f9          stable             1191          23.1  0.0      1514.0
content_a5a2fbc76336          stable              307          39.8  0.0      1342.0


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.